# 大规模数据工程：从来源清单到可训练 Shard

> **本章定位**：将 Dataset 与 Tokenizer 契约扩展到生产规模，覆盖异构数据集分类与 Schema 适配、来源治理、质量与隐私策略、去重、污染检测、稳定划分、混合、Packing、Sharding 和血缘。

> **章节边界**：本章属于模型训练与适配：数据工程专题，以 `20`、`21` 和 `40` 的数据与训练契约为基础，负责异构训练数据的规模化接入、校验、规范化和版本化构建；SFT、DPO、GRPO 的损失语义见 `41`，推理轨迹训练与验证器见 `A30`，分布式执行引擎的部署细节不在本章范围内。

**本章总览**：以不可变异构来源快照为输入，先声明物理格式、样本 Schema、消费用途与学习目标、内容属性、模态及治理血缘，经版本化适配器生成类型化记录，再完成质量治理、去重、污染隔离、稳定划分和目标相关的 Token 化处理，输出 Dataset Manifest、训练 Block Shard、质量报告与删除血缘。

```mermaid
flowchart LR
    S["Heterogeneous Source Snapshot"] --> M["Source + Dataset Contract"]
    M --> F["Decode + Schema Validate"]
    F --> E["Canonicalize + Policy Route"]
    E --> Q["Quality / Language / PII"]
    Q --> D["Exact + Near Dedup"]
    D --> C["Contamination Filter"]
    C --> P["Group-stable Split + Mixture"]
    P --> T["Template + Pinned Tokenizer + Label Policy"]
    T --> K["Pack / Shard"]
    K --> R["Dataset Manifest + Quality Report"]
```


## 1．学习契约

| 项目 | 内容 |
|---|---|
| 路线 | 模型训练与适配：数据工程专题 |
| 本章定位 | 纵向放大数据资产，把异构格式与小型 Dataset/Tokenizer 契约扩展到分布式生产。 |
| 先修知识 | 掌握 `20`、`21` 的数据与分词契约，以及 `40` 的训练目标和数据消费知识。 |
| 预计时间 | 2～3 小时 |
| 运行资源 | 最小流程 CPU 可运行；真实构建需要对象存储与分布式执行引擎。 |
| 输入 | 带物理格式、Schema、消费用途、来源、授权、记录/分组身份和准确命名哈希的异构原始快照。 |
| 交付物 | 版本化 Dataset Contract、Canonical Record、Manifest、去重/污染证据、训练 Block Shard 和血缘。 |

### 1.1．学习目标

完成本章后，读者能够区分物理格式、样本 Schema、消费用途与学习目标、内容属性、模态、治理与血缘六个维度，使用版本化 Schema Registry 将异构来源校验并规范化为类型化记录，设计可审计的数据 Manifest，实施质量与隐私治理、精确及近似去重、污染隔离和稳定划分，并生成可追溯、可恢复、可增量构建的训练 Block Shard。


## 2．直觉与输入输出契约

### 2.1．数据集分类的六个维度

“数据集类型”不是单一枚举。一份数据需要同时声明以下六个维度；把不同维度的术语并列，会让加载器、校验器和训练消费者无法形成稳定契约。前五项描述数据及其消费者，第六项是贯穿所有类型的治理与血缘属性集合。

| 维度 | 回答的问题 | 典型取值 | 进入 Manifest 的关键字段 |
|---|---|---|---|
| 物理格式 | 字节如何编码、压缩、分区与读取 | 纯文本、JSON/JSONL、CSV/TSV、Parquet、Arrow IPC、TAR Shard | `container_format`、`compression`、`encoding`、`partitioning` |
| 样本 Schema | 一条逻辑样本由哪些字段、角色和原子边界组成 | 文档、平行句对、Alpaca 指令、ShareGPT/OpenAI Messages、偏好对、RL Prompt、推理轨迹 | `schema_name`、`schema_version`、`sample_type`、`record_id_key`、`split_group_key` |
| 消费用途与学习目标 | 哪个消费者以何种方式使用样本 | Tokenizer 训练、预训练、SFT、Reward Model/DPO、GRPO/RLVR、蒸馏、评测 | `consumer_objective`、`label_policy`、`reward_or_verifier_revision` |
| 内容与能力属性 | 数据承载什么语言、领域、任务和能力 | 通用文本、对话、代码、数学、CoT/推理轨迹、工具调用、安全任务、领域知识 | `content_tags`、`language`、`domain` |
| 模态 | 样本携带哪些媒体及其同步关系 | 文本、图像、音频、视频、混合模态 | `modalities`、`media_manifest`、`processor_revision` |
| 治理与血缘 | 数据从何而来、是否允许使用、经过哪些策略 | 来源快照、许可证、同意状态、PII、风险、质量、保留期限、删除链路 | `source_manifest`、`license_policy`、`risk_tags`、`quality_policy`、`lineage` |

因此，`JSONL` 是物理格式，`Alpaca` 是指令—补全字段约定，`ShareGPT` 与 OpenAI Messages 是对话消息约定，“对话数据集”描述交互结构或内容族，`CoT` 描述内容与监督粒度。它们不属于同一层级，也不能互相替代。CoT 可以存放在 JSONL、Parquet 或 Arrow IPC 中，也可以采用指令、消息或独立推理轨迹 Schema。JSON 与 CSV 也只描述容器：CSV/TSV 适合扁平记录，但不宜直接承载嵌套消息、工具调用或多模态内容块。WebDataset 也不只是 TAR 扩展名，而是建立在 TAR Shard 之上的样本键、分组和读取约定，因此其容器与逻辑分组契约需要分别记录。

| 常见称呼 | 正确层级 | 精确定义 | 不能据此推断 |
|---|---|---|---|
| JSONL 数据集 | 物理格式 | 每行一个 JSON 值，适合交换、流式落地与逐行恢复 | 不能推断是预训练、SFT、对话或 CoT |
| Alpaca 数据集 | 样本 Schema | 通常使用 `instruction`、可选 `input` 和 `output` | 不等于全部 SFT，也不等于 JSONL |
| ShareGPT 数据集 | 样本 Schema | 通常使用 `conversations` 与 `from/value` 表示有序消息 | 不保证角色、工具调用和多模态语义与目标模型兼容 |
| OpenAI Messages 数据集 | 样本 Schema | 使用有序 `messages` 与 `role/content`，可扩展工具和多模态内容块 | 不自动决定 Chat Template、Label Mask 或消费用途 |
| 对话数据集 | 交互结构与内容属性 | 单轮或多轮角色消息，可能包含 System、Tool 与 Observation | 不限定 ShareGPT 或 OpenAI 字段，也不必包含 CoT |
| CoT/推理数据集 | 内容与监督粒度 | 包含推理轨迹、最终答案及可选步骤标签或验证证据 | 不存在通用“CoT 文件格式”或“CoT Loss” |

### 2.2．逻辑 Schema、数据消费者与章节边界

| Canonical Sample Type | 最小逻辑字段 | 主要消费者 | 本课程的展开位置 |
|---|---|---|---|
| `document_text` | `text`、来源与文档身份 | Tokenizer、继续预训练 | `20`、`21`、`40` |
| `parallel_pair` | `source`、`target`、语言对与组身份 | 翻译训练、跨语言 SFT | `20`、`30` |
| `sft_instruction` | `instruction`、可选 `input`、`completion` | SFT | `41` |
| `sft_chat` | 有序 `messages`、角色、内容与会话身份 | 单轮/多轮对话 SFT | `41`、`42` |
| `tool_trajectory` | 消息、Tool Schema、Call/Result 关联 ID 与步骤预算 | Tool-use SFT、Agent 评测 | `90` |
| `preference_pair` | 同一 Prompt 下的 `chosen`、`rejected` | Reward Model、DPO | `41` |
| `rl_prompt` | Prompt、可选参考答案、Reward/Verifier 输入 | GRPO、RLVR 在线 Rollout | `41`、`A30` |
| `reasoning_trace` | 问题、轨迹、最终答案、步骤标签、验证器版本 | Trace SFT、PRM、验证后蒸馏 | `A30` |
| `multimodal_messages` | 消息、媒体引用/内容块及 Processor 契约 | 多模态预训练或 SFT | `E20` |
| `evaluation_case` | 输入、期望结果或 Rubric、Slice、风险级别且不进入训练 | 离线评测与发布门禁 | `50`、`70` |

同一个上游文件可以物化为不同训练视图，但每个发布版本只能绑定明确的消费者契约。例如推理样本可以物化为只监督最终答案的 SFT 视图、监督轨迹与答案的 Trace SFT 视图，或仅保留 Prompt 与验证器输入的 RLVR 视图；三者的 Label、泄漏风险和评测口径不同，不能共用含糊的 `type=chat` 标签。

### 2.3．版本化适配器与判别联合

生产内部采用“公共 Envelope + 按 `sample_type` 判别的类型化 Payload”，而不是把所有字段塞入一张大部分为空的万能宽表。Envelope 固定记录身份、防泄漏分组身份、Schema 版本、来源、许可证、语言、模态、内容标签，以及分别命名的源对象/源文本/已解析输入/Canonical Payload 哈希；Payload 分别保存文档、消息、偏好对或推理轨迹。Alpaca、ShareGPT、OpenAI Messages 等入口格式由显式版本化适配器转换，原始记录与适配器 Revision 保留在血缘中。

未知或歧义 Schema 进入隔离区并记录原因码，不能根据几个同名字段静默猜测。自动猜错会改变消息顺序、Chosen/Rejected 配对、工具调用边界或 Reasoning/Final 的监督范围，这类错误通常能正常完成训练，却会产生不可解释的模型行为。

<!-- diagram:versioned-schema-adapter-architecture -->

![架构图：版本化 Schema Adapter 将物理输入转换为 Canonical 数据产品并隔离异常](assets/figures/A20_data_engineering/versioned-schema-adapter-architecture.svg)

[TikZ 源文件](assets/figures/A20_data_engineering/versioned-schema-adapter-architecture.tex)

### 2.4．控制面与数据面

![架构图：数据工程控制面编排数据面并汇聚血缘、指标与重试状态](assets/figures/A20_data_engineering/data-control-planes.svg)

[TikZ 源文件](assets/figures/A20_data_engineering/data-control-planes.tex)

数据面处理字节、类型化样本和 Token；控制面决定 Reader、Schema Registry、策略、调度、重试、血缘和发布。校验失败记录连同稳定原因码进入隔离区。把两者混在一个巨型脚本中，会导致局部失败只能全量重跑，也无法解释某个 Checkpoint 使用了哪一批数据、采用了哪一个 Schema 与监督边界。


In [ ]:
import hashlib
import json
import math
import errno
import os
import platform
import re
import tempfile
import unicodedata
from collections import Counter, defaultdict
from collections.abc import Mapping
from dataclasses import asdict, dataclass, replace
from datetime import datetime, timezone
from pathlib import Path
from types import MappingProxyType


from tqdm.auto import tqdm

@dataclass(frozen=True)
class MyDatasetContract:
    """显式绑定物理格式、逻辑 Schema、消费用途、内容、模态与治理策略。"""
    dataset_id: str
    container_format: str
    schema_name: str
    schema_version: str
    sample_type: str
    consumer_objective: str
    record_id_key: str
    split_group_key: str
    label_policy: str
    content_tags: tuple[str, ...]
    modalities: tuple[str, ...]
    language: str
    license_policy: str
    adapter_revision: str
    quality_policy: str
    compression: str = "none"
    text_encoding: str = "utf-8"
    partitioning: str = "unpartitioned"
    risk_tags: tuple[str, ...] = ()
    reward_or_verifier_revision: str | None = None
    processor_revision: str | None = None


dataset_contracts = {
    "web-corpus-v1": MyDatasetContract(
        "web-corpus-v1", "jsonl", "document_text", "v1", "document_text",
        "pretraining", "document_id", "document_id", "next_token", ("general_text",),
        ("text",), "zh", "mixed-source-license-policy@1", "document_text-adapter@1",
        "document-quality-policy@1",
    ),
    "alpaca-sft-v1": MyDatasetContract(
        "alpaca-sft-v1", "jsonl", "alpaca", "v1", "sft_instruction",
        "sft", "sample_id", "sample_id", "completion_only", ("instruction_following",),
        ("text",), "zh", "cc-by-4.0-policy@1", "alpaca-adapter@1",
        "instruction-quality-policy@1",
    ),
    "sharegpt-chat-v1": MyDatasetContract(
        "sharegpt-chat-v1", "parquet", "sharegpt_text", "v1", "sft_chat",
        "sft", "conversation_id", "conversation_id", "assistant_messages", ("dialogue",),
        ("text",), "zh", "opt-in-policy@1", "sharegpt-text-adapter@1",
        "conversation-quality-policy@1",
        compression="zstd", partitioning="language/sample_type",
    ),
    "openai-chat-v1": MyDatasetContract(
        "openai-chat-v1", "jsonl", "openai_text_messages", "v1", "sft_chat",
        "sft", "conversation_id", "conversation_id", "assistant_messages", ("dialogue",),
        ("text",), "zh", "internal-training-policy@1", "openai-text-messages-adapter@1",
        "conversation-quality-policy@1",
    ),
    "preference-v1": MyDatasetContract(
        "preference-v1", "parquet", "preference_pair", "v1", "preference_pair",
        "dpo", "pair_id", "pair_id", "chosen_rejected", ("preference",),
        ("text",), "zh", "internal-training-policy@1", "preference-adapter@1",
        "preference-quality-policy@1",
        compression="zstd", partitioning="language/sample_type",
    ),
    "verified-reasoning-v1": MyDatasetContract(
        "verified-reasoning-v1", "parquet", "reasoning_trace", "v1", "reasoning_trace",
        "verified_distillation", "trace_id", "problem_id", "trace_and_final", ("reasoning", "math"),
        ("text",), "zh", "internal-training-policy@1", "reasoning-trace-adapter@1",
        "reasoning-quality-policy@1",
        compression="zstd", partitioning="language/sample_type",
        risk_tags=("sensitive_reasoning_trace",),
        reward_or_verifier_revision="arithmetic-verifier@1",
    ),
}


@dataclass(frozen=True)
class MySourceDocument:
    """描述一条带来源、获取时间、许可证、语种和正文的数据源文档。"""
    document_id: str
    source_name: str
    source_uri: str
    retrieved_at: str
    license_id: str
    language: str
    text: str


RETRIEVED_AT = datetime(2026, 8, 8, tzinfo=timezone.utc).isoformat()
documents = [
    MySourceDocument("web-001", "licensed_web", "https://example.org/a", RETRIEVED_AT, "CC-BY-4.0", "zh", "Transformer 使用注意力在 Token 之间传递信息。"),
    MySourceDocument("web-002", "licensed_web", "https://example.org/b", RETRIEVED_AT, "CC-BY-4.0", "zh", "Transformer 使用注意力在 token 之间传递信息。"),
    MySourceDocument("docs-001", "product_docs", "s3://docs/train.md", RETRIEVED_AT, "INTERNAL-TRAINING", "zh", "训练制品必须绑定数据、Tokenizer、代码和配置版本。"),
    MySourceDocument("docs-002", "product_docs", "s3://docs/deploy.md", RETRIEVED_AT, "INTERNAL-TRAINING", "zh", "部署前必须完成容量、质量与安全验收。联系 owner@example.com。"),
    MySourceDocument("forum-001", "opt_in_forum", "https://forum.example/1", RETRIEVED_AT, "OPT-IN", "zh", "KV Cache 复用历史 Token 的键和值，从而减少 Decode 重复计算。"),
]

source_manifest = [
    {
        "document_id": item.document_id,
        "source_name": item.source_name,
        "source_uri": item.source_uri,
        "retrieved_at": item.retrieved_at,
        "license_id": item.license_id,
        "source_text_sha256": hashlib.sha256(item.text.encode("utf-8")).hexdigest(),
    }
    for item in documents
]
source_manifest


<!-- theory-math-contract:v1 -->
### 2.5．核心机制的语言与数学表达

Packing 的目标是在不破坏样本原子边界和 Attention/Label 语义的前提下提高固定长度块的 Token 利用率：

$$
U_{\mathrm{pack}}=\frac{\sum_{i=1}^{N}L_i}{K\,L_{\mathrm{block}}},\qquad
K=\left\lceil\frac{\sum_i L_i+W}{L_{\mathrm{block}}}\right\rceil
$$

其中，$L_i$ 是第 $i$ 个样本的有效 Token 数，$L_{\mathrm{block}}$ 是块长，$K$ 是物化块数，$W$ 是分隔符及不可利用空位带来的额外 Token。`packer` 计算块布局，`attention_mask` 与 `labels` 保留边界语义，Shard Manifest 记录每片 Token 量。高利用率不等价于训练正确；跨样本可见性、EOS、Label Mask 和分布式 Shard 均衡必须分别验证。

以上数学表示用于明确变量、形状与约束；实际结论仍需由本章的数值、形状、梯度、性能或失败案例证据验证。

## 3．最小原理实现

### 3.1．Schema 校验与规范化路由

本节输入是原始记录和显式 `MyDatasetContract`，输出是公共 Envelope 与类型化 Payload 组成的 `MyCanonicalRecord`，或带稳定原因码的隔离记录。Registry Key 由 `schema_name/schema_version` 构成；适配器只能处理已经声明的 Schema，不能根据字段猜测类型。

最小实现覆盖纯文档、Alpaca、ShareGPT 文本消息、OpenAI 文本 Messages、偏好对和推理轨迹六类入口。它展示校验与路由原理，不替代 JSON Schema、Pydantic、Arrow Schema 或生产 Schema Registry。两个文本消息 Schema 仅接受非空字符串内容并保持原有顺序；工具调用、结构化内容块与多模态消息需要独立的判别联合和 Schema Version，不能由文本适配器部分解析。偏好对保持 Prompt/Chosen/Rejected 原子性；推理轨迹将 `reasoning_trace`、`final_answer` 与 `verifier_revision` 分开保存。平行句对、RL Prompt 与多模态适配器分别回链 `20/30`、`41/A30` 与 `E20`，本节不重复实现。用于后续治理与 Shard 构建的五条来源文档也经过同一个 `document_text/v1` Adapter，后续阶段不再绕过 Canonical Record。


In [ ]:
@dataclass(frozen=True)
class MyCanonicalRecord:
    """公共 Envelope 与按 sample_type 判别的类型化 Payload。"""
    record_id: str
    split_group_id: str
    sample_type: str
    payload: Mapping[str, object]
    metadata: Mapping[str, object]
    lineage: Mapping[str, str]


class MySchemaValidationError(ValueError):
    """表示可稳定归因并进入隔离区的 Schema 错误。"""


def my_deep_freeze(value: object) -> object:
    """递归冻结 Mapping 与序列，避免 Frozen Dataclass 内部仍可原位修改。"""
    if isinstance(value, dict):
        return MappingProxyType({key: my_deep_freeze(item) for key, item in value.items()})
    if isinstance(value, (list, tuple)):
        return tuple(my_deep_freeze(item) for item in value)
    return value


def my_required_text(raw_record: dict[str, object], field: str) -> str:
    """读取必需的非空字符串，不把数值或容器静默转成文本。"""
    value = raw_record.get(field)
    if not isinstance(value, str) or not value.strip():
        raise MySchemaValidationError(f"required_non_empty_text:{field}")
    return value.strip()


def my_adapt_document_text(raw_record: dict[str, object]) -> dict[str, object]:
    return {"text": my_required_text(raw_record, "text")}


def my_adapt_alpaca(raw_record: dict[str, object]) -> dict[str, object]:
    instruction = my_required_text(raw_record, "instruction")
    completion = my_required_text(raw_record, "output")
    input_text = raw_record.get("input", "")
    if not isinstance(input_text, str):
        raise MySchemaValidationError("expected_text:input")
    return {"instruction": instruction, "input": input_text, "completion": completion}


SHAREGPT_ROLE_MAP = {
    "human": ("user", "message"),
    "gpt": ("assistant", "message"),
    "system": ("system", "message"),
}


def my_adapt_sharegpt(raw_record: dict[str, object]) -> dict[str, object]:
    conversations = raw_record.get("conversations")
    if not isinstance(conversations, list) or not conversations:
        raise MySchemaValidationError("required_non_empty_list:conversations")
    messages = []
    for index, item in enumerate(conversations):
        if not isinstance(item, dict):
            raise MySchemaValidationError(f"expected_object:conversations[{index}]")
        source_role = item.get("from")
        if not isinstance(source_role, str) or source_role not in SHAREGPT_ROLE_MAP:
            raise MySchemaValidationError(f"unsupported_role:conversations[{index}]")
        content = my_required_text(item, "value")
        role, message_kind = SHAREGPT_ROLE_MAP[source_role]
        messages.append({"role": role, "kind": message_kind, "content": content})
    if not any(message["role"] == "assistant" for message in messages):
        raise MySchemaValidationError("missing_assistant_message")
    return {"messages": messages}


def my_adapt_openai_messages(raw_record: dict[str, object]) -> dict[str, object]:
    source_messages = raw_record.get("messages")
    allowed_roles = {"system", "user", "assistant"}
    if not isinstance(source_messages, list) or not source_messages:
        raise MySchemaValidationError("required_non_empty_list:messages")
    messages = []
    for index, item in enumerate(source_messages):
        if not isinstance(item, dict) or not isinstance(item.get("role"), str) or item.get("role") not in allowed_roles:
            raise MySchemaValidationError(f"invalid_role:messages[{index}]")
        content = my_required_text(item, "content")
        messages.append({"role": item["role"], "kind": "message", "content": content})
    if not any(message["role"] == "assistant" for message in messages):
        raise MySchemaValidationError("missing_assistant_message")
    return {"messages": messages}


def my_adapt_preference_pair(raw_record: dict[str, object]) -> dict[str, object]:
    prompt = my_required_text(raw_record, "prompt")
    chosen = my_required_text(raw_record, "chosen")
    rejected = my_required_text(raw_record, "rejected")
    if chosen == rejected:
        raise MySchemaValidationError("identical_preference_responses")
    return {"prompt": prompt, "chosen": chosen, "rejected": rejected}


def my_adapt_reasoning_trace(raw_record: dict[str, object]) -> dict[str, object]:
    trace = raw_record.get("reasoning_trace")
    if not isinstance(trace, list) or not trace or not all(isinstance(step, str) and step.strip() for step in trace):
        raise MySchemaValidationError("required_non_empty_text_list:reasoning_trace")
    step_labels = raw_record.get("step_labels")
    if step_labels is not None and (
        not isinstance(step_labels, list)
        or len(step_labels) != len(trace)
        or not all(isinstance(label, bool) for label in step_labels)
    ):
        raise MySchemaValidationError("invalid_step_labels")
    return {
        "problem": my_required_text(raw_record, "problem"),
        "reasoning_trace": [step.strip() for step in trace],
        "final_answer": my_required_text(raw_record, "final_answer"),
        "step_labels": step_labels,
        "verifier_revision": my_required_text(raw_record, "verifier_revision"),
    }


SCHEMA_REGISTRY = {
    ("document_text", "v1"): ("document_text", my_adapt_document_text),
    ("alpaca", "v1"): ("sft_instruction", my_adapt_alpaca),
    ("sharegpt_text", "v1"): ("sft_chat", my_adapt_sharegpt),
    ("openai_text_messages", "v1"): ("sft_chat", my_adapt_openai_messages),
    ("preference_pair", "v1"): ("preference_pair", my_adapt_preference_pair),
    ("reasoning_trace", "v1"): ("reasoning_trace", my_adapt_reasoning_trace),
}
SCHEMA_REGISTRY_REVISION = "canonical-schema-registry@2"
SUPPORTED_CONTAINER_FORMATS = {
    "text", "json", "jsonl", "csv", "tsv", "parquet", "arrow_ipc", "tar"
}


def my_stable_json_bytes(value: object) -> bytes:
    """序列化已解析对象的确定性表示；它不是源对象原始字节哈希。"""
    return json.dumps(
        value, ensure_ascii=False, sort_keys=True, separators=(",", ":")
    ).encode("utf-8")


def my_optional_sha256(raw_record: dict[str, object], field: str) -> str:
    """读取可选的源字节/文本哈希，并拒绝名不副实的值。"""
    value = raw_record.get(field, "")
    if not isinstance(value, str) or (value and (len(value) != 64 or any(char not in "0123456789abcdef" for char in value.casefold()))):
        raise MySchemaValidationError(f"invalid_sha256:{field}")
    return value.casefold()


def my_validate_and_canonicalize(
    raw_record: dict[str, object], contract: MyDatasetContract
) -> MyCanonicalRecord:
    """按显式 Contract 校验一条记录，并生成带血缘的类型化记录。"""
    if contract.container_format not in SUPPORTED_CONTAINER_FORMATS:
        raise MySchemaValidationError("unsupported_container_format")
    registry_key = (contract.schema_name, contract.schema_version)
    if registry_key not in SCHEMA_REGISTRY:
        raise MySchemaValidationError("unknown_schema_registry_key")
    expected_sample_type, adapter = SCHEMA_REGISTRY[registry_key]
    if contract.sample_type != expected_sample_type:
        raise MySchemaValidationError("contract_sample_type_mismatch")
    record_id = my_required_text(raw_record, contract.record_id_key)
    split_group_id = my_required_text(raw_record, contract.split_group_key)
    payload = adapter(raw_record)
    if (
        contract.reward_or_verifier_revision is not None
        and payload.get("verifier_revision") != contract.reward_or_verifier_revision
    ):
        raise MySchemaValidationError("verifier_revision_mismatch")
    source_text_sha256 = my_optional_sha256(raw_record, "source_text_sha256")
    if contract.sample_type == "document_text" and source_text_sha256:
        actual_source_text_sha256 = hashlib.sha256(
            str(payload["text"]).encode(contract.text_encoding)
        ).hexdigest()
        if actual_source_text_sha256 != source_text_sha256:
            raise MySchemaValidationError("source_text_sha256_mismatch")
    input_record_bytes = my_stable_json_bytes(raw_record)
    canonical_payload_bytes = my_stable_json_bytes(payload)
    source_uri = raw_record.get("source_uri", f"memory://{contract.dataset_id}/{record_id}")
    if not isinstance(source_uri, str) or not source_uri:
        raise MySchemaValidationError("invalid_source_uri")
    return MyCanonicalRecord(
        record_id=record_id,
        split_group_id=split_group_id,
        sample_type=contract.sample_type,
        payload=my_deep_freeze(payload),
        metadata=my_deep_freeze({
            "dataset_id": contract.dataset_id,
            "container_format": contract.container_format,
            "compression": contract.compression,
            "text_encoding": contract.text_encoding,
            "partitioning": contract.partitioning,
            "consumer_objective": contract.consumer_objective,
            "label_policy": contract.label_policy,
            "content_tags": contract.content_tags,
            "risk_tags": contract.risk_tags,
            "modalities": contract.modalities,
            "language": contract.language,
            "license_policy": contract.license_policy,
            "source_name": str(raw_record.get("source_name", "not_declared")),
            "source_license_id": str(raw_record.get("license_id", "not_declared")),
            "retrieved_at": str(raw_record.get("retrieved_at", "not_declared")),
            "quality_policy": contract.quality_policy,
            "reward_or_verifier_revision": contract.reward_or_verifier_revision,
            "processor_revision": contract.processor_revision,
        }),
        lineage=my_deep_freeze({
            "source_uri": source_uri,
            "source_record_locator": str(raw_record.get("source_record_locator", record_id)),
            "source_object_sha256": my_optional_sha256(raw_record, "source_object_sha256"),
            "source_text_sha256": source_text_sha256,
            "source_schema": f"{contract.schema_name}/{contract.schema_version}",
            "schema_registry_revision": SCHEMA_REGISTRY_REVISION,
            "adapter_revision": contract.adapter_revision,
            "input_record_sha256": hashlib.sha256(input_record_bytes).hexdigest(),
            "canonical_payload_sha256": hashlib.sha256(canonical_payload_bytes).hexdigest(),
        }),
    )


def my_canonicalize_or_quarantine(
    raw_record: dict[str, object], contract: MyDatasetContract
) -> tuple[MyCanonicalRecord | None, dict[str, str] | None]:
    """成功时返回 Canonical Record，失败时返回无原文泄漏的隔离原因。"""
    source_uri = raw_record.get("source_uri", f"memory://{contract.dataset_id}/unknown")
    try:
        input_record_sha256 = hashlib.sha256(my_stable_json_bytes(raw_record)).hexdigest()
    except (TypeError, ValueError):
        return None, {
            "dataset_id": contract.dataset_id,
            "record_id": str(raw_record.get(contract.record_id_key, "<missing>")),
            "split_group_id": str(raw_record.get(contract.split_group_key, "<missing>")),
            "source_uri": str(source_uri),
            "input_record_sha256": "",
            "schema_registry_revision": SCHEMA_REGISTRY_REVISION,
            "adapter_revision": contract.adapter_revision,
            "reason_code": "INPUT_NOT_JSON_SERIALIZABLE",
            "detail": "input_record_not_json_serializable",
        }
    try:
        return my_validate_and_canonicalize(raw_record, contract), None
    except MySchemaValidationError as error:
        return None, {
            "dataset_id": contract.dataset_id,
            "record_id": str(raw_record.get(contract.record_id_key, "<missing>")),
            "split_group_id": str(raw_record.get(contract.split_group_key, "<missing>")),
            "source_uri": str(source_uri),
            "input_record_sha256": input_record_sha256,
            "schema_registry_revision": SCHEMA_REGISTRY_REVISION,
            "adapter_revision": contract.adapter_revision,
            "reason_code": "SCHEMA_VALIDATION_FAILED",
            "detail": str(error),
        }


raw_schema_samples = [
    (dataset_contracts["web-corpus-v1"], {"document_id": "doc-001", "text": "版本化数据需要可追溯血缘。"}),
    (dataset_contracts["alpaca-sft-v1"], {"sample_id": "inst-001", "instruction": "解释数据血缘。", "input": "", "output": "数据血缘记录资产的来源与变换。"}),
    (dataset_contracts["sharegpt-chat-v1"], {"conversation_id": "chat-001", "conversations": [{"from": "human", "value": "什么是 Shard？"}, {"from": "gpt", "value": "Shard 是可独立读取与校验的数据分片。"}]}),
    (dataset_contracts["openai-chat-v1"], {"conversation_id": "chat-002", "messages": [{"role": "user", "content": "什么是 Schema？"}, {"role": "assistant", "content": "Schema 定义字段、类型与约束。"}]}),
    (dataset_contracts["preference-v1"], {"pair_id": "pref-001", "prompt": "如何发布数据？", "chosen": "先校验，再原子发布并写入 Manifest。", "rejected": "直接覆盖线上文件。"}),
    (dataset_contracts["verified-reasoning-v1"], {"trace_id": "reason-001", "problem_id": "problem-001", "problem": "计算 7 + 5。", "reasoning_trace": ["从 7 开始。", "加 5 得到 12。"], "final_answer": "12", "step_labels": [True, True], "verifier_revision": "arithmetic-verifier@1"}),
    (dataset_contracts["alpaca-sft-v1"], {"sample_id": "invalid-001", "instruction": "", "output": "缺少有效指令。"}),
]

canonical_records = []
quarantine_records = []
for contract, raw_record in tqdm(
    raw_schema_samples, desc="规范化异构 Schema", unit="record", dynamic_ncols=True
):
    canonical_record, quarantine_record = my_canonicalize_or_quarantine(raw_record, contract)
    if canonical_record is not None:
        canonical_records.append(canonical_record)
    else:
        quarantine_records.append(quarantine_record)


def my_index_canonical_records(
    records: list[MyCanonicalRecord],
) -> dict[tuple[str, str], MyCanonicalRecord]:
    """以 Dataset Namespace + Record ID 建立唯一索引，重复时显式失败。"""
    index = {}
    for record in records:
        key = (str(record.metadata["dataset_id"]), record.record_id)
        if key in index:
            raise MySchemaValidationError("duplicate_namespaced_record_id")
        index[key] = record
    return index


canonical_record_index = my_index_canonical_records(canonical_records)

source_manifest_by_document_id = {row["document_id"]: row for row in source_manifest}
source_document_by_id = {document.document_id: document for document in documents}
corpus_raw_records = [
    {
        "document_id": document.document_id,
        "text": document.text,
        "source_name": document.source_name,
        "source_uri": document.source_uri,
        "source_record_locator": document.document_id,
        "retrieved_at": document.retrieved_at,
        "license_id": document.license_id,
        "source_text_sha256": source_manifest_by_document_id[document.document_id]["source_text_sha256"],
    }
    for document in documents
]
corpus_canonical_records = []
corpus_quarantine_records = []
for raw_record in tqdm(
    corpus_raw_records, desc="规范化语料 Envelope", unit="record", dynamic_ncols=True
):
    canonical_record, quarantine_record = my_canonicalize_or_quarantine(
        raw_record, dataset_contracts["web-corpus-v1"]
    )
    if canonical_record is not None:
        corpus_canonical_records.append(canonical_record)
    else:
        corpus_quarantine_records.append(quarantine_record)

corpus_canonical_record_index = my_index_canonical_records(corpus_canonical_records)
canonical_documents = [
    replace(
        source_document_by_id[record.record_id],
        text=str(record.payload["text"]),
    )
    for record in corpus_canonical_records
]

{
    "canonical_types": Counter(record.sample_type for record in canonical_records),
    "quarantine_reason_codes": Counter(row["reason_code"] for row in quarantine_records),
    "corpus_canonical_records": len(corpus_canonical_records),
    "corpus_quarantine_records": len(corpus_quarantine_records),
}


### 3.2．抽取、规范化与 PII 处理

解析器负责从 HTML、PDF、Office、代码或日志中恢复结构；规范化负责稳定 Unicode 和空白；PII 阶段负责检测、分级、删除、替换或隔离。单一正则不足以承担完整隐私治理；本节正则用于展示可审计变换与统计契约。对对话、偏好对和推理轨迹执行文本变换时，需要逐字段处理并保持角色顺序、配对关系、步骤标签和媒体引用，不能先拼成一个字符串再尝试恢复结构。


In [ ]:
# 邮箱域后缀至少 2 个字符，仅覆盖本章最小 PII 规则；语种、法规或误删/漏删分布变化时重审。
EMAIL_PATTERN = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
ALLOWED_LICENSE_IDS = {"CC-BY-4.0", "INTERNAL-TRAINING", "OPT-IN"}
LICENSE_POLICY_REVISION = "mixed-source-license-policy@1"


def my_normalize_text(text: str) -> str:
    # NFKC 与“3 个以上连续换行压为 2 个”属于版本化文本协议；代码、表格或语言分布变化时用保真样本复核。
    """按版本化规则执行 NFKC、空白压缩和首尾清理，返回规范化文本。"""
    normalized = unicodedata.normalize("NFKC", text)
    normalized = re.sub(r"[ \t]+", " ", normalized)
    normalized = re.sub(r"\n{3,}", "\n\n", normalized)
    return normalized.strip()


def my_redact_email(text: str) -> tuple[str, int]:
    """将匹配到的邮箱替换为占位符，并返回脱敏文本与事件数量。"""
    matches = EMAIL_PATTERN.findall(text)
    return EMAIL_PATTERN.sub("<EMAIL_REDACTED>", text), len(matches)


normalized_documents = []
governance_rejections = []
pii_event_count = 0
for document in tqdm(
    canonical_documents, desc="执行治理与 PII 处理", unit="document", dynamic_ncols=True
):
    if document.license_id not in ALLOWED_LICENSE_IDS:
        governance_rejections.append(
            {"document_id": document.document_id, "reason_code": "LICENSE_NOT_ALLOWED"}
        )
        continue
    normalized = my_normalize_text(document.text)
    redacted, events = my_redact_email(normalized)
    pii_event_count += events
    normalized_documents.append(replace(document, text=redacted))

{
    "documents": len(normalized_documents),
    "pii_events": pii_event_count,
    "governance_reason_codes": Counter(
        row["reason_code"] for row in governance_rejections
    ),
}


### 3.3．质量评分与原因码

生产质量通常组合规则、分类模型和来源先验。质量特征需要按 `sample_type` 定义：文档关注正文质量，对话还要检查角色交替与轮次完整性，偏好对检查配对一致性与长度/风格偏差，推理轨迹检查答案、步骤标签和 Verifier 覆盖。建议保存字符数、语言置信度、符号比例、重复行、广告/模板概率、困惑度区间和策略版本。删除原因需要支持按 Schema、来源和内容标签聚合，否则无法知道某次规则升级改变了什么。


In [ ]:
# 阈值只服务于本章冻结短文本；语种、来源或长度分布变化时按人工标注集重新校准。
MIN_QUALITY_CHARACTERS = 12
MIN_QUALITY_ALNUM_RATIO = 0.50
MAX_REPEATED_LINE_RATIO = 0.50
QUALITY_POLICY_REVISION = "document-quality-policy@1"


def my_quality_features(text: str) -> dict[str, float]:
    """计算字符数、可见字符中的字母数字比例和重复行比例。"""
    characters = len(text)
    visible = sum(not char.isspace() for char in text)
    letters_or_numbers = sum(char.isalnum() for char in text)
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    repeated_lines = len(lines) - len(set(lines))
    return {
        "characters": float(characters),
        "alnum_ratio": letters_or_numbers / max(visible, 1),
        "repeated_line_ratio": repeated_lines / max(len(lines), 1),
    }


def my_quality_decision(features: dict[str, float]) -> dict[str, object]:
    """根据冻结阈值返回可聚合的分数、准入状态和稳定原因码。"""
    reason_codes = []
    if features["characters"] < MIN_QUALITY_CHARACTERS:
        reason_codes.append("TOO_SHORT")
    if features["alnum_ratio"] < MIN_QUALITY_ALNUM_RATIO:
        reason_codes.append("LOW_ALNUM_RATIO")
    if features["repeated_line_ratio"] > MAX_REPEATED_LINE_RATIO:
        reason_codes.append("REPEATED_LINES")
    quality_score = (
        0.40 * min(features["characters"] / 40.0, 1.0)
        + 0.40 * features["alnum_ratio"]
        + 0.20 * (1.0 - features["repeated_line_ratio"])
    )
    return {
        "quality_score": quality_score,
        "accepted": not reason_codes,
        "reason_codes": tuple(reason_codes),
        "policy_revision": QUALITY_POLICY_REVISION,
    }


quality_rows = [
    {
        "document_id": document.document_id,
        **(features := my_quality_features(document.text)),
        **my_quality_decision(features),
    }
    for document in tqdm(
        normalized_documents, desc="计算质量特征", unit="document", dynamic_ncols=True
    )
]
quality_row_by_id = {row["document_id"]: row for row in quality_rows}
quality_accepted_documents = [
    document for document in normalized_documents
    if quality_row_by_id[document.document_id]["accepted"]
]
{
    "accepted": len(quality_accepted_documents),
    "reason_codes": Counter(
        reason_code
        for row in quality_rows
        for reason_code in row["reason_codes"]
    ),
}


### 3.4．精确去重与近似去重

精确去重对规范化内容计算哈希；近似去重先生成 Shingle，再用 MinHash 压缩集合，最后用 LSH 只产生候选对，候选仍需计算真实相似度。对话、偏好对和推理轨迹以会话、配对或问题家族为原子，字段级相似度用于发现候选，删除决策作用于完整原子记录并保留来源映射。直接全量两两比较是平方复杂度，无法扩展到数十亿文档。


In [ ]:
# 5-gram 适配本章短中文文本；减小会提高召回和误报，语种或长度分布变化时重新标定。
SHINGLE_SIZE = 5
# 32 维签名拆为 8×4 的 LSH；Hash 数增大提高稳定性但增加 CPU/存储，乘积必须等于签名长度。
NUM_HASHES = 32
BANDS = 8
ROWS_PER_BAND = 4
if NUM_HASHES != BANDS * ROWS_PER_BAND:
    raise ValueError("NUM_HASHES 必须等于 BANDS × ROWS_PER_BAND")
# 0.82 是近重复与污染的发布初始阈值；按标注对的召回/误删曲线及 Benchmark 版本复核。
NEAR_DUPLICATE_THRESHOLD = 0.82


def my_content_hash(text: str) -> str:
    """对规范化且大小写折叠后的文本计算 SHA-256 内容指纹。"""
    canonical = my_normalize_text(text).casefold()
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()


def my_shingles(text: str, size: int = SHINGLE_SIZE) -> set[str]:
    """移除空白后切分固定长度字符 Shingle，短文本至少返回一个片段。"""
    canonical = re.sub(r"\s+", "", my_normalize_text(text).casefold())
    return {canonical[index:index + size] for index in range(max(len(canonical) - size + 1, 1))}


# seed 只是确定性哈希函数序号；它不代表训练随机性或质量优势。
def my_seeded_hash(value: str, seed: int) -> int:
    # 4-Byte 前缀与 8-Byte BLAKE2b 摘要定义签名协议；种子集合变化需全量重建数据版本。
    """用种子前缀和 BLAKE2b 生成确定性的 64 位 Shingle 哈希。"""
    payload = seed.to_bytes(4, "big") + value.encode("utf-8")
    return int.from_bytes(hashlib.blake2b(payload, digest_size=8).digest(), "big")


def my_minhash_signature(shingles: set[str], num_hashes: int = NUM_HASHES) -> tuple[int, ...]:
    """对每个种子取全部 Shingle 的最小哈希，返回固定长度 MinHash 签名。"""
    return tuple(min(my_seeded_hash(shingle, seed) for shingle in shingles) for seed in range(num_hashes))


def my_lsh_candidates(signatures: dict[str, tuple[int, ...]]) -> set[tuple[str, str]]:
    """按 Band 将 MinHash 签名分桶，返回可能近重复的文档 ID 对。"""
    buckets = defaultdict(list)
    for document_id, signature in signatures.items():
        for band in range(BANDS):
            start = band * ROWS_PER_BAND
            band_key = (band, signature[start:start + ROWS_PER_BAND])
            buckets[band_key].append(document_id)
    candidates = set()
    for document_ids in buckets.values():
        for left_index in range(len(document_ids)):
            for right_index in range(left_index + 1, len(document_ids)):
                candidates.add(tuple(sorted((document_ids[left_index], document_ids[right_index]))))
    return candidates


def my_jaccard(left: set[str], right: set[str]) -> float:
    """返回两个 Shingle 集合的 Jaccard 相似度，并安全处理空并集。"""
    return len(left & right) / max(len(left | right), 1)


# Survivor Key 固定来源准入优先级、质量分、时间和 Record ID；分布式输入重排不会改变保留对象。
SOURCE_SURVIVOR_PRIORITY = {
    "product_docs": 0,
    "licensed_web": 1,
    "opt_in_forum": 2,
}
DEDUP_POLICY_REVISION = "content-dedup-policy@2"


def my_survivor_key(document: MySourceDocument) -> tuple[object, ...]:
    """返回与分区及输入顺序无关的稳定全序保留键。"""
    quality_score = float(quality_row_by_id[document.document_id]["quality_score"])
    return (
        SOURCE_SURVIVOR_PRIORITY.get(document.source_name, 1_000),
        -quality_score,
        document.retrieved_at,
        document.document_id,
    )


documents_by_content_hash = defaultdict(list)
for document in tqdm(
    quality_accepted_documents, desc="计算精确内容哈希", unit="document", dynamic_ncols=True
):
    documents_by_content_hash[my_content_hash(document.text)].append(document)

unique_by_hash = {}
exact_duplicate_of = {}
for digest, duplicate_group in sorted(documents_by_content_hash.items()):
    survivor = min(duplicate_group, key=my_survivor_key)
    unique_by_hash[digest] = survivor
    for document in duplicate_group:
        if document.document_id != survivor.document_id:
            exact_duplicate_of[document.document_id] = survivor.document_id

exact_unique = sorted(unique_by_hash.values(), key=lambda document: document.document_id)
shingle_sets = {item.document_id: my_shingles(item.text) for item in exact_unique}
signatures = {
    key: my_minhash_signature(value)
    for key, value in tqdm(
        shingle_sets.items(), total=len(shingle_sets), desc="计算 MinHash 签名",
        unit="document", dynamic_ncols=True,
    )
}
candidates = my_lsh_candidates(signatures)
near_duplicate_pairs = [
    {"left": left, "right": right, "jaccard": my_jaccard(shingle_sets[left], shingle_sets[right])}
    for left, right in sorted(candidates)
    if my_jaccard(shingle_sets[left], shingle_sets[right]) >= NEAR_DUPLICATE_THRESHOLD
]
{"exact_duplicate_of": exact_duplicate_of, "near_duplicates": near_duplicate_pairs}


#### 3.4.1．去重关系与稳定保留对象

**学习问题**：规范化后的文档中，哪些记录被判为精确重复，哪些候选通过真实 Jaccard 阈值成为近似重复关系，最终稳定保留的是哪一份？

**验收不变量**：`exact_duplicate_of` 的源与目标必须都来自质量准入文档，目标必须是重复组内 `my_survivor_key` 的最小值；精确去重减少的文档数必须等于映射条数；每条近似重复边的 Jaccard 必须不低于 `NEAR_DUPLICATE_THRESHOLD`。


In [ ]:
# 直接消费精确去重映射和通过阈值复核的近似重复对，呈现稳定保留关系。
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

dedup_document_ids = [document.document_id for document in quality_accepted_documents]
dedup_position = {document_id: index for index, document_id in enumerate(dedup_document_ids)}
dedup_removed_ids = set(exact_duplicate_of)
fig, axis = plt.subplots(figsize=(12, 4.5))
for document_id, x_position in dedup_position.items():
    is_removed = document_id in dedup_removed_ids
    axis.scatter(
        x_position,
        0,
        s=900,
        color="#E69F00" if is_removed else "#009E73",
        edgecolor="black",
        zorder=3,
    )
    axis.text(x_position, 0, document_id, ha="center", va="center", fontsize=8, zorder=4)

for duplicate_id, retained_id in exact_duplicate_of.items():
    duplicate_x = dedup_position[duplicate_id]
    retained_x = dedup_position[retained_id]
    axis.annotate(
        "",
        xy=(retained_x, 0.08),
        xytext=(duplicate_x, 0.08),
        arrowprops={
            "arrowstyle": "->",
            "color": "#D55E00",
            "linewidth": 2,
            "connectionstyle": "arc3,rad=0.35",
        },
    )
    axis.text((duplicate_x + retained_x) / 2, 0.48, "精确重复", ha="center", color="#D55E00")

for pair_index, pair in enumerate(near_duplicate_pairs):
    left_x = dedup_position[pair["left"]]
    right_x = dedup_position[pair["right"]]
    curve = -0.25 - 0.08 * pair_index
    axis.annotate(
        "",
        xy=(right_x, -0.08),
        xytext=(left_x, -0.08),
        arrowprops={
            "arrowstyle": "-",
            "linestyle": "--",
            "color": "#0072B2",
            "linewidth": 1.8,
            "connectionstyle": f"arc3,rad={curve}",
        },
    )
    axis.text(
        (left_x + right_x) / 2,
        -0.48 - 0.08 * pair_index,
        f"Jaccard={pair['jaccard']:.2f}",
        ha="center",
        color="#0072B2",
    )

axis.set_xlim(-0.7, len(dedup_document_ids) - 0.3)
axis.set_ylim(-1.1, 1.0)
axis.set_title("规范化文档的去重关系：箭头指向稳定保留记录")
axis.set_axis_off()
axis.legend(
    handles=[
        Line2D([0], [0], marker="o", color="none", markerfacecolor="#009E73", markeredgecolor="black", markersize=12, label="保留记录"),
        Line2D([0], [0], marker="o", color="none", markerfacecolor="#E69F00", markeredgecolor="black", markersize=12, label="精确重复记录"),
        Line2D([0], [0], color="#D55E00", linewidth=2, label="精确重复映射"),
        Line2D([0], [0], color="#0072B2", linestyle="--", linewidth=2, label="阈值复核后的近似重复"),
    ],
    loc="lower center",
    ncol=4,
)
fig.tight_layout()
plt.show()

dedup_known_ids = set(dedup_document_ids)
dedup_relationship_checks = {
    "exact_edges_reference_inputs": all(
        duplicate_id in dedup_known_ids and retained_id in dedup_known_ids
        for duplicate_id, retained_id in exact_duplicate_of.items()
    ),
    "stable_survivor_key_wins": all(
        my_survivor_key(source_document_by_id[retained_id])
        < my_survivor_key(source_document_by_id[duplicate_id])
        for duplicate_id, retained_id in exact_duplicate_of.items()
    ),
    "exact_count_reconciles": (
        len(quality_accepted_documents) - len(exact_unique) == len(exact_duplicate_of)
    ),
    "near_edges_meet_threshold": all(
        pair["jaccard"] >= NEAR_DUPLICATE_THRESHOLD for pair in near_duplicate_pairs
    ),
}
print(dedup_relationship_checks)


**应观察结论**：规范化与 `casefold` 消除了 `web-001` 和 `web-002` 的 Token 大小写差异；两条记录的来源优先级、质量和获取时间相同时，稳定 Record ID 使 `web-002` 指向保留的 `web-001`，结果不依赖分区到达顺序。蓝色虚线仅表示 LSH 召回后又通过真实 Jaccard 阈值的关系。

**不可误读边界**：LSH 候选不是删除判据，图中没有近似重复边也不能证明不存在漏召回；文本重复也不表示来源、许可证和血缘可以合并丢弃，被合并记录仍需保留来源映射与审计信息。


### 3.5．污染检测与评测隔离

评测题、参考答案及其改写进入训练集会制造虚假能力。污染检测应对 Benchmark 建立受控指纹，按策略扫描 Prompt、Completion、Chosen/Rejected、最终答案和受限推理字段，比较精确哈希、长片段、Shingle 和语义候选；命中记录进入隔离区，由数据治理者决定删除、降权或标记。Benchmark 本身也要版本化并限制访问，扫描结果不得反向泄露受控答案。


In [ ]:
BENCHMARK_REVISION = "deployment-gate-benchmark@1"
CONTAMINATION_POLICY_REVISION = "benchmark-contamination-policy@1"
benchmark_items = {
    "eval-001": "部署前必须完成容量、质量与安全验收。联系 <EMAIL_REDACTED>。",
}
benchmark_shingles = {key: my_shingles(value) for key, value in benchmark_items.items()}

contamination_hits = []
for document in tqdm(
    exact_unique, desc="扫描评测集污染", unit="document", dynamic_ncols=True
):
    document_shingles = shingle_sets[document.document_id]
    for benchmark_id, eval_shingles in benchmark_shingles.items():
        similarity = my_jaccard(document_shingles, eval_shingles)
        if similarity >= NEAR_DUPLICATE_THRESHOLD:
            contamination_hits.append(
                {"document_id": document.document_id, "benchmark_id": benchmark_id, "similarity": similarity}
            )
contamination_hits


### 3.6．稳定划分与来源混合

Split 应使用 Contract 中的 `split_group_key`：文档使用文档家族，对话使用会话或用户边界，偏好数据使用完整 Pair，推理数据使用问题/轨迹家族，多模态数据还要绑定媒体身份；不能在 Chunk、消息、候选回答或媒体帧之后随机划分。混合权重常用温度平滑：

$$
p_i = \frac{n_i^\alpha}{\sum_j n_j^\alpha}
$$

当 `α < 1` 时，小来源相对权重提高。质量未达标、污染命中或许可证不允许的来源不进入混合池；降低采样权重不能替代准入治理。本章发布一次性、无重复采样的训练 Block，因此同时记录目标温度权重与实际物化权重，不把尚未应用的目标权重伪装成 Shard 的实际分布。若训练 Loader 采用带替换的温度采样，需要使用按来源可寻址的 Shard、固定采样预算和 Sampler Revision，并形成另一个训练视图版本。


In [ ]:
# alpha=0.7 相对抬高小来源；趋近 0 更均匀、趋近 1 更接近按规模采样，来源变化时重做能力消融。
MIXTURE_TEMPERATURE = 0.7
MIXTURE_POLICY_REVISION = "temperature-mixture-plan@1"


# 10,000 个稳定桶实现 98%/1%/1% 划分；比例增大评估集会提高稳定性但减少训练量，边界变更须发布新数据版本。
def my_deterministic_split(split_group_id: str) -> str:
    """将防泄漏分组身份稳定映射到 train、validation 或 test。"""
    bucket = int(hashlib.sha256(split_group_id.encode("utf-8")).hexdigest()[:8], 16) % 10_000
    if bucket < 9_800:
        return "train"
    if bucket < 9_900:
        return "validation"
    return "test"


def my_temperature_mixture(source_counts: dict[str, int], alpha: float) -> dict[str, float]:
    """对各来源样本数施加温度幂并归一化为采样权重。"""
    if not math.isfinite(alpha) or not 0.0 < alpha <= 1.0:
        raise ValueError("当前混合契约要求有限且位于 (0, 1] 的温度指数")
    if not source_counts or any(count <= 0 for count in source_counts.values()):
        raise ValueError("来源计数必须为非空正整数")
    adjusted = {source: count ** alpha for source, count in source_counts.items()}
    denominator = sum(adjusted.values())
    return {source: value / denominator for source, value in adjusted.items()}


quarantined_ids = {item["document_id"] for item in contamination_hits}
split_rows = [
    {
        "document_id": document.document_id,
        "split_group_id": corpus_canonical_record_index[("web-corpus-v1", document.document_id)].split_group_id,
        "split": my_deterministic_split(
            corpus_canonical_record_index[("web-corpus-v1", document.document_id)].split_group_id
        ),
    }
    for document in exact_unique
]
split_by_document_id = {row["document_id"]: row["split"] for row in split_rows}
eligible_train_documents = [
    document for document in exact_unique
    if split_by_document_id[document.document_id] == "train"
    and document.document_id not in quarantined_ids
]
eligible_source_counts = Counter(document.source_name for document in eligible_train_documents)
target_mixture_weights = my_temperature_mixture(
    dict(eligible_source_counts), MIXTURE_TEMPERATURE
)
eligible_single_pass_document_weights = {
    source: count / max(sum(eligible_source_counts.values()), 1)
    for source, count in eligible_source_counts.items()
}
{
    "eligible_source_counts": eligible_source_counts,
    "target_mixture_weights": target_mixture_weights,
    "eligible_single_pass_document_weights": eligible_single_pass_document_weights,
    "splits": split_rows,
}


### 3.7．Tokenizer、Packing 与 Sharding 契约

Tokenized 数据需要先通过目标相关的 Materializer：对话固定 Chat Template 与角色映射，偏好对对 Chosen/Rejected 使用同一 Prompt 和截断策略，推理数据固定 Reasoning/Final 的 Label Policy，多模态数据固定 Processor 与媒体占位符。其后再记录 Tokenizer ID、文件哈希、特殊 Token 协议、Block Size、是否跨样本 Packing、EOS 插入规则和 Label Mask。Packing 不能破坏会话、偏好对、工具轨迹或媒体同步边界。本章把完整训练 Block 与 Canonical 来源 Span 一起写入构建级临时目录，全部校验后再将目录原子发布到由冻结构建规格导出的 Build Spec ID；已发布目录只校验和复用，不原位覆盖，Shard 内容则由各自 SHA-256 校验。


In [ ]:
import transformers
from transformers import AutoTokenizer

TOKENIZER_ID = "google/byt5-small"
PIPELINE_REVISION = "a20-canonical-block-builder@2"
NORMALIZATION_POLICY_REVISION = "nfkc-whitespace-normalization@1"
PII_POLICY_REVISION = "email-redaction@1"
SPLIT_POLICY_REVISION = "sha256-group-split-9800-100-100@1"
PACKING_POLICY_REVISION = "cross-document-eos-full-block-drop-tail@1"
# 128 Token Block 对应小模型上下文；增大可提高长程利用但增加激活和截断风险，须与模型上下文联调。
BLOCK_SIZE = 128
# 256 Token/Shard 等于两个完整 Block，仅用于生成多个可检查制品；生产按读取吞吐与恢复粒度调优。
TOKENS_PER_SHARD = 256
if TOKENS_PER_SHARD % BLOCK_SIZE != 0:
    raise ValueError("TOKENS_PER_SHARD 必须是 BLOCK_SIZE 的整数倍")
BLOCKS_PER_SHARD = TOKENS_PER_SHARD // BLOCK_SIZE

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)
if tokenizer.eos_token_id is None:
    raise ValueError("当前 Packing 协议要求 Tokenizer 提供 EOS Token")

train_documents = eligible_train_documents

token_stream = []
token_source_spans = []
for document in tqdm(
    train_documents, desc="Tokenize 训练文档", unit="document", dynamic_ncols=True
):
    canonical_record = corpus_canonical_record_index[("web-corpus-v1", document.document_id)]
    document_tokens = tokenizer.encode(document.text, add_special_tokens=False)
    document_tokens.append(tokenizer.eos_token_id)
    span_start = len(token_stream)
    token_stream.extend(document_tokens)
    span_end = len(token_stream)
    token_source_spans.append(
        {
            "dataset_id": str(canonical_record.metadata["dataset_id"]),
            "record_id": canonical_record.record_id,
            "split_group_id": canonical_record.split_group_id,
            "source_name": document.source_name,
            "stream_start": span_start,
            "stream_end": span_end,
            "input_record_sha256": canonical_record.lineage["input_record_sha256"],
            "canonical_payload_sha256": canonical_record.lineage["canonical_payload_sha256"],
            "materialized_text_sha256": hashlib.sha256(
                document.text.encode("utf-8")
            ).hexdigest(),
            "dedup_content_sha256": my_content_hash(document.text),
        }
    )

# 只物化完整 Block；尾部丢弃量、跨文档 EOS 和 Label 语义进入 Loader Contract。
blocks = []
block_records = []
block_starts = range(0, len(token_stream) - BLOCK_SIZE + 1, BLOCK_SIZE)
for block_index, block_start in enumerate(tqdm(
    block_starts, desc="物化固定长度 Block", unit="block", dynamic_ncols=True
)):
    block_end = block_start + BLOCK_SIZE
    input_ids = token_stream[block_start:block_end]
    source_spans = []
    for source_span in token_source_spans:
        overlap_start = max(block_start, source_span["stream_start"])
        overlap_end = min(block_end, source_span["stream_end"])
        if overlap_start < overlap_end:
            source_spans.append(
                {
                    **{
                        key: value for key, value in source_span.items()
                        if key not in {"stream_start", "stream_end"}
                    },
                    "block_start": overlap_start - block_start,
                    "block_end": overlap_end - block_start,
                }
            )
    blocks.append(input_ids)
    block_records.append(
        {
            "block_id": f"block-{block_index:08d}",
            "input_ids": input_ids,
            # Causal LM 在模型内部右移；Prompt-free 预训练视图中 labels 与 input_ids 相同。
            "labels": input_ids.copy(),
            "source_spans": source_spans,
        }
    )

if not block_records:
    raise RuntimeError("冻结样本不足以形成一个完整训练 Block")

packed_full_block_tokens = len(block_records) * BLOCK_SIZE
packing_tail_tokens = len(token_stream) - packed_full_block_tokens
persisted_token_counts_by_source = Counter()
for source_span in token_source_spans:
    persisted = max(
        0,
        min(source_span["stream_end"], packed_full_block_tokens)
        - source_span["stream_start"],
    )
    persisted_token_counts_by_source[source_span["source_name"]] += persisted
actual_materialized_token_weights = {
    source: count / max(packed_full_block_tokens, 1)
    for source, count in persisted_token_counts_by_source.items()
}

special_tokens_map = {
    key: [str(item) for item in value] if isinstance(value, list) else str(value)
    for key, value in tokenizer.special_tokens_map.items()
}
build_spec = {
    "pipeline_revision": PIPELINE_REVISION,
    "runtime_fingerprint": {
        "python": platform.python_version(),
        "transformers": transformers.__version__,
        "unicode_database": unicodedata.unidata_version,
    },
    "dataset_contract": asdict(dataset_contracts["web-corpus-v1"]),
    "schema_registry_revision": SCHEMA_REGISTRY_REVISION,
    "source_manifest_sha256": hashlib.sha256(my_stable_json_bytes(source_manifest)).hexdigest(),
    "canonical_payload_sha256s": sorted(
        record.lineage["canonical_payload_sha256"] for record in corpus_canonical_records
    ),
    "policy_revisions": {
        "normalization": NORMALIZATION_POLICY_REVISION,
        "pii": PII_POLICY_REVISION,
        "license": LICENSE_POLICY_REVISION,
        "quality": QUALITY_POLICY_REVISION,
        "dedup": DEDUP_POLICY_REVISION,
        "contamination": CONTAMINATION_POLICY_REVISION,
        "benchmark": BENCHMARK_REVISION,
        "split": SPLIT_POLICY_REVISION,
        "mixture": MIXTURE_POLICY_REVISION,
        "packing": PACKING_POLICY_REVISION,
    },
    "policy_parameters": {
        "allowed_license_ids": sorted(ALLOWED_LICENSE_IDS),
        "pii_email_pattern": EMAIL_PATTERN.pattern,
        "quality": {
            "min_characters": MIN_QUALITY_CHARACTERS,
            "min_alnum_ratio": MIN_QUALITY_ALNUM_RATIO,
            "max_repeated_line_ratio": MAX_REPEATED_LINE_RATIO,
        },
        "dedup": {
            "shingle_size": SHINGLE_SIZE,
            "num_hashes": NUM_HASHES,
            "bands": BANDS,
            "rows_per_band": ROWS_PER_BAND,
            "near_duplicate_threshold": NEAR_DUPLICATE_THRESHOLD,
            "source_survivor_priority": SOURCE_SURVIVOR_PRIORITY,
        },
        "contamination": {
            "benchmark_items_sha256": hashlib.sha256(
                my_stable_json_bytes(benchmark_items)
            ).hexdigest(),
            "similarity_threshold": NEAR_DUPLICATE_THRESHOLD,
        },
        "split_bucket_boundaries": {
            "train_end_exclusive": 9_800,
            "validation_end_exclusive": 9_900,
            "bucket_count": 10_000,
        },
        "mixture_temperature": MIXTURE_TEMPERATURE,
    },
    "tokenizer_id": TOKENIZER_ID,
    "special_tokens_map": special_tokens_map,
    "eos_token_id": tokenizer.eos_token_id,
    "block_size": BLOCK_SIZE,
    "tokens_per_shard": TOKENS_PER_SHARD,
    "target_mixture_weights": target_mixture_weights,
}
build_spec_id = hashlib.sha256(my_stable_json_bytes(build_spec)).hexdigest()
dataset_version = f"large-scale-corpus-{build_spec_id}"
builds_root = Path("artifacts/data/large_scale_corpus/builds")
builds_root.mkdir(parents=True, exist_ok=True)
artifact_dir = builds_root / build_spec_id
candidate_release_manifest_sha256 = None

if not artifact_dir.exists():
    with tempfile.TemporaryDirectory(prefix=f".{build_spec_id}-", dir=builds_root) as temporary_dir:
        staging_dir = Path(temporary_dir)
        staged_shard_manifest = []
        shard_starts = range(0, len(block_records), BLOCKS_PER_SHARD)
        for shard_index, start in enumerate(tqdm(
            shard_starts, desc="写入 Dataset Shard", unit="shard", dynamic_ncols=True
        )):
            shard_blocks = block_records[start:start + BLOCKS_PER_SHARD]
            shard_name = f"train-{shard_index:05d}.json"
            shard_path = staging_dir / shard_name
            shard_payload = {
                "schema_version": "causal_lm_training_blocks/v1",
                "blocks": shard_blocks,
            }
            shard_path.write_bytes(my_stable_json_bytes(shard_payload))
            shard_record_keys = [
                {"dataset_id": dataset_id, "record_id": record_id}
                for dataset_id, record_id in sorted({
                    (span["dataset_id"], span["record_id"])
                    for block in shard_blocks
                    for span in block["source_spans"]
                })
            ]
            staged_shard_manifest.append(
                {
                    "path": shard_name,
                    "blocks": len(shard_blocks),
                    "tokens": len(shard_blocks) * BLOCK_SIZE,
                    "record_keys": shard_record_keys,
                    "sha256": hashlib.sha256(shard_path.read_bytes()).hexdigest(),
                }
            )

        manifest = {
            "dataset_version": dataset_version,
            "build_spec_id": build_spec_id,
            "artifact_kind": "causal_lm_training_block_shards",
            "build_spec": build_spec,
            "source_manifest": source_manifest,
            "canonical_lineage": [
                {
                    "record_id": record.record_id,
                    "split_group_id": record.split_group_id,
                    **dict(record.lineage),
                }
                for record in corpus_canonical_records
            ],
            "governance_report": {
                "pii_event_count": pii_event_count,
                "schema_quarantine_reason_counts": dict(Counter(
                    row["reason_code"] for row in corpus_quarantine_records
                )),
                "governance_rejection_reason_counts": dict(Counter(
                    row["reason_code"] for row in governance_rejections
                )),
                "record_level_audit_sha256": hashlib.sha256(my_stable_json_bytes({
                    "schema_quarantine": corpus_quarantine_records,
                    "governance_rejections": governance_rejections,
                })).hexdigest(),
                "record_level_audit_storage": "external_access_controlled_registry_required",
            },
            "quality_report": {
                "policy_revision": QUALITY_POLICY_REVISION,
                "reason_counts": dict(Counter(
                    reason_code for row in quality_rows for reason_code in row["reason_codes"]
                )),
                "records": quality_rows,
            },
            "exact_duplicate_of": exact_duplicate_of,
            "near_duplicate_pairs": near_duplicate_pairs,
            "contamination_hits": contamination_hits,
            "splits": split_rows,
            "mixture": {
                "materialization_mode": "single_pass_without_replacement",
                "eligible_source_counts": dict(eligible_source_counts),
                "target_weights_not_applied": target_mixture_weights,
                "eligible_document_weights": eligible_single_pass_document_weights,
                "actual_persisted_token_weights": actual_materialized_token_weights,
            },
            "loader_contract": {
                "schema_version": "causal_lm_training_blocks/v1",
                "labels": "equal_input_ids_model_applies_next_token_shift",
                "cross_document_packing": True,
                "document_separator_token_id": tokenizer.eos_token_id,
                "tail_policy": "drop_incomplete_final_block",
                "dropped_tail_tokens": packing_tail_tokens,
                "source_span_coordinates": "half_open_block_offsets",
            },
            "shards": staged_shard_manifest,
        }
        manifest["release_manifest_sha256"] = hashlib.sha256(
            my_stable_json_bytes(manifest)
        ).hexdigest()
        candidate_release_manifest_sha256 = manifest["release_manifest_sha256"]
        manifest_path_in_staging = staging_dir / "manifest.json"
        manifest_path_in_staging.write_text(
            json.dumps(manifest, ensure_ascii=False, indent=2, sort_keys=True),
            encoding="utf-8",
        )
        if sum(item["tokens"] for item in staged_shard_manifest) != packed_full_block_tokens:
            raise RuntimeError("发布前 Shard Token 数与完整 Block 不一致")
        try:
            os.replace(staging_dir, artifact_dir)
        except OSError as exc:
            # 并发构建中仅接受“同一 Build Spec 的胜者已原子发布”这一种竞态。
            if exc.errno not in {errno.EEXIST, errno.ENOTEMPTY} or not artifact_dir.is_dir():
                raise

manifest_path = artifact_dir / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
if manifest["build_spec"] != build_spec:
    raise RuntimeError("已发布 Manifest 的构建规格与当前构建规格不一致")
if manifest["build_spec_id"] != build_spec_id:
    raise RuntimeError("构建规格寻址目录与 Manifest Build Spec ID 不一致")
release_manifest_sha256 = manifest.get("release_manifest_sha256")
manifest_payload = {
    key: value for key, value in manifest.items() if key != "release_manifest_sha256"
}
if hashlib.sha256(my_stable_json_bytes(manifest_payload)).hexdigest() != release_manifest_sha256:
    raise RuntimeError("Release Manifest 摘要校验失败")
if (
    candidate_release_manifest_sha256 is not None
    and candidate_release_manifest_sha256 != release_manifest_sha256
):
    raise RuntimeError("并发发布胜者与本地候选制品不一致")
for shard in tqdm(
    manifest["shards"], desc="校验 Dataset Shard", unit="shard", dynamic_ncols=True
):
    shard_path = artifact_dir / shard["path"]
    if hashlib.sha256(shard_path.read_bytes()).hexdigest() != shard["sha256"]:
        raise RuntimeError(f"Shard 哈希校验失败：{shard['path']}")
shard_manifest = manifest["shards"]
manifest


#### 3.7.1．Block Packing 利用率与 Shard Token 分布

**学习问题**：同一 `token_stream` 物化为完整训练 Block 后使用了多少 Token、尾部丢弃多少，构建规格寻址的 Shard 又如何只分配这些完整 Block？

**验收不变量**：完整 Block Token 与尾部 Token 之和必须等于 `token_stream` 长度；每个 `blocks` 元素必须恰好为 `BLOCK_SIZE`，且 Block 内来源 Span 恰好覆盖全部位置；Manifest 中各 Shard Token 数之和必须等于完整 Block Token 数，单个 Shard 不得超过 `TOKENS_PER_SHARD`，并且只能包含整数个 Block。


In [ ]:
# 直接消费 token_stream、完整 Block 与已发布 Shard Manifest，核对 Packing 和持久化边界。
import matplotlib.pyplot as plt

packed_full_block_tokens = len(blocks) * BLOCK_SIZE
packing_tail_tokens = len(token_stream) - packed_full_block_tokens
shard_labels = [Path(item["path"]).name for item in shard_manifest]
shard_token_counts = [item["tokens"] for item in shard_manifest]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
axes[0].barh(
    ["token_stream"],
    [packed_full_block_tokens],
    color="#0072B2",
    label=f"完整 Block（{len(blocks)}×{BLOCK_SIZE}）",
)
axes[0].barh(
    ["token_stream"],
    [packing_tail_tokens],
    left=[packed_full_block_tokens],
    color="#E69F00",
    label="不足一个 Block 的尾部",
)
if packed_full_block_tokens:
    axes[0].text(
        packed_full_block_tokens / 2, 0, str(packed_full_block_tokens),
        ha="center", va="center", color="white",
    )
if packing_tail_tokens:
    axes[0].text(
        packed_full_block_tokens + packing_tail_tokens / 2, 0, str(packing_tail_tokens),
        ha="center", va="center", color="black",
    )
axes[0].set_xlim(0, max(len(token_stream), 1) * 1.05)
axes[0].set_xlabel("Token 数")
axes[0].set_title("训练 Block：只消费完整的固定长度块")
axes[0].legend(loc="lower center", bbox_to_anchor=(0.5, -0.35), ncol=2)
axes[0].grid(axis="x", alpha=0.25)

if shard_token_counts:
    shard_bars = axes[1].bar(
        range(len(shard_token_counts)), shard_token_counts, color="#009E73"
    )
    axes[1].set_xticks(range(len(shard_labels)))
    axes[1].set_xticklabels(shard_labels, rotation=25, ha="right")
    for bar, token_count in zip(shard_bars, shard_token_counts):
        axes[1].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            str(token_count),
            ha="center",
            va="bottom",
        )
else:
    axes[1].text(
        0.5, 0.5, "当前 Train Split 没有可发布 Token",
        transform=axes[1].transAxes, ha="center", va="center",
    )
    axes[1].set_xticks([])
axes[1].axhline(
    TOKENS_PER_SHARD, color="#D55E00", linestyle="--", label="Shard Token 上限"
)
axes[1].set_ylabel("Token 数")
axes[1].set_title("训练 Block Shard：最后一个 Shard 可不足上限")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.25)
fig.suptitle("Token 流的完整 Block 物化与构建规格寻址 Shard", fontsize=14)
fig.tight_layout()
plt.show()

def my_source_spans_cover_block(source_spans: list[dict[str, object]]) -> bool:
    """验证半开区间从 0 连续覆盖整个 Block，既无缺口也无重叠。"""
    ordered = sorted(source_spans, key=lambda span: int(span["block_start"]))
    return (
        bool(ordered)
        and int(ordered[0]["block_start"]) == 0
        and int(ordered[-1]["block_end"]) == BLOCK_SIZE
        and all(int(span["block_start"]) < int(span["block_end"]) for span in ordered)
        and all(
            int(left["block_end"]) == int(right["block_start"])
            for left, right in zip(ordered, ordered[1:])
        )
    )


packing_shard_checks = {
    "packing_reconciles": (
        packed_full_block_tokens + packing_tail_tokens == len(token_stream)
    ),
    "all_blocks_are_full": all(len(block) == BLOCK_SIZE for block in blocks),
    "block_source_spans_reconcile": all(
        my_source_spans_cover_block(block["source_spans"])
        for block in block_records
    ),
    "shards_reconcile_full_blocks": (
        sum(shard_token_counts) == packed_full_block_tokens
    ),
    "shards_respect_limit": all(
        token_count <= TOKENS_PER_SHARD for token_count in shard_token_counts
    ),
    "shards_contain_only_full_blocks": all(
        token_count % BLOCK_SIZE == 0 for token_count in shard_token_counts
    ),
    "manifest_has_hashes": all(bool(item["sha256"]) for item in shard_manifest),
    "source_spans_distinguish_materialized_and_dedup_hashes": all(
        {"materialized_text_sha256", "dedup_content_sha256"} <= set(span)
        for block in block_records for span in block["source_spans"]
    ),
    "build_spec_address_matches_manifest": (
        artifact_dir.name == manifest["build_spec_id"]
    ),
    "release_manifest_digest_matches": (
        hashlib.sha256(my_stable_json_bytes(manifest_payload)).hexdigest()
        == release_manifest_sha256
    ),
    "manifest_records_governance_evidence": (
        manifest["governance_report"]["pii_event_count"] == pii_event_count
        and len(manifest["governance_report"]["record_level_audit_sha256"]) == 64
    ),
    "build_spec_freezes_effective_parameters": (
        manifest["build_spec"]["policy_parameters"]["mixture_temperature"]
        == MIXTURE_TEMPERATURE
        and manifest["build_spec"]["runtime_fingerprint"]["transformers"]
        == transformers.__version__
    ),
    "target_mixture_is_normalized": math.isclose(
        sum(target_mixture_weights.values()), 1.0, rel_tol=0.0, abs_tol=1e-12
    ),
    "persisted_token_weights_are_normalized": math.isclose(
        sum(actual_materialized_token_weights.values()), 1.0, rel_tol=0.0, abs_tol=1e-12
    ),
    "manifest_distinguishes_target_and_actual_mixture": (
        manifest["mixture"]["target_weights_not_applied"] == target_mixture_weights
        and manifest["mixture"]["actual_persisted_token_weights"]
        == actual_materialized_token_weights
    ),
    "shards_have_structured_lineage_keys": all(
        all(set(record_key) == {"dataset_id", "record_id"} for record_key in shard["record_keys"])
        for shard in shard_manifest
    ),
}
print(packing_shard_checks)


**应观察结论**：左图把同一 Token 流分为已物化的完整训练 Block 和按冻结策略丢弃的尾部；右图显示 Manifest 中每个构建规格寻址 Shard 的实际 Token 数，除最后一个 Shard 外通常达到目标上限。Shard 中保存的是带 `input_ids`、`labels` 与 Canonical 来源 Span 的完整 Block，不再把平坦 Token Stream 与训练 Block 混为同一种制品。

**不可误读边界**：尾部 Token 在当前版本中明确不进入训练 Shard；改变为 Padding、跨构建 Carry-over 或可变长 Block 都会形成新的 Loader Contract 和数据版本。Shard Token 数接近也不表示来源分布、样本质量、训练计算时间或压缩后字节数完全相同，生产系统仍需分别监控有效 Label、来源权重、字节大小和读取吞吐。


## 4．证据验证

流水线验证同时覆盖 Schema 语义、内容语义、集合隔离与制品一致性。Schema 阶段核对每条输入只进入 Canonical 或 Quarantine 之一、声明类型与 Registry 一致、消息顺序和偏好/轨迹原子字段不丢失、来源定位、已解析输入哈希、Canonical Payload 哈希与 Adapter Revision 完整；正文可见时还要重算源文本哈希，Verifier Revision 必须与 Contract 一致。规范化和 PII 处理保留变换前后计数及原因码；去重阶段报告候选召回、真实 Jaccard 与误删样本；Split 阶段检查内容家族交集；Packing 与 Sharding 阶段核对有效 Token、连续无重叠的来源 Span、尾部处理、文件摘要和 Manifest。源对象字节哈希、源文本哈希、已解析记录哈希、实际 Tokenizer 输入文本哈希与去重规范哈希分别命名，不能都写成含糊的 `raw_sha256`。

下面的数值证据应全部为 `True`：7 条格式样本恰好形成 6 条 Canonical Record 与 1 条隔离记录；两个对话样本保持 User → Assistant 顺序；偏好对与推理轨迹保留各自的判别字段；推理样本的唯一 `record_id` 与用于防泄漏划分的 `split_group_id` 相互独立；五条真实来源文档全部经过 Canonical 路径；每条成功记录均携带输入与 Payload 的 64 位十六进制 SHA-256。该结果只能证明本章冻结样本和 Registry 的契约一致，不能证明所有第三方 Alpaca/ShareGPT 变体都兼容。

固定输入重复构建时，样本路由、分组划分、Token 序列、Block 来源 Span、Shard 顺序、Build Spec ID 和哈希应保持一致。目标混合权重与实际持久化 Token 权重分别归一化并分别报告；构建目录名必须等于 Manifest 的 Build Spec ID，各 Shard 字节和 Release Manifest 必须通过 SHA-256 校验。Schema、Adapter、策略、Tokenizer、阈值或尾部处理发生变化时，需要生成新的数据版本并重新执行下游验收。Release Manifest 摘要用于完整性检查；生产中的真实性还需由外部制品目录固定该摘要，或使用 KMS 签名与透明日志，不能信任制品目录内可被一并改写的自声明摘要。


In [ ]:
# 直接消费 3.1 的 Canonical 与 Quarantine 结果，核对路由、原子字段和血缘。
sharegpt_record = canonical_record_index[("sharegpt-chat-v1", "chat-001")]
openai_record = canonical_record_index[("openai-chat-v1", "chat-002")]
preference_record = canonical_record_index[("preference-v1", "pref-001")]
reasoning_record = canonical_record_index[("verified-reasoning-v1", "reason-001")]
sharegpt_roles = [
    message["role"] for message in sharegpt_record.payload["messages"]
]
openai_roles = [
    message["role"] for message in openai_record.payload["messages"]
]
preference_payload = preference_record.payload
reasoning_payload = reasoning_record.payload
split_membership_by_group = defaultdict(set)
for row in split_rows:
    split_membership_by_group[row["split_group_id"]].add(row["split"])

schema_contract_checks = {
    "all_inputs_accounted_for": (
        len(canonical_records) + len(quarantine_records) == len(raw_schema_samples)
    ),
    "declared_types_match_registry": all(
        record.sample_type
        == dataset_contracts[record.metadata["dataset_id"]].sample_type
        for record in canonical_records
    ),
    "sharegpt_order_preserved": sharegpt_roles == ["user", "assistant"],
    "openai_order_preserved": openai_roles == ["user", "assistant"],
    "canonical_payload_is_deep_frozen": (
        isinstance(sharegpt_record.payload, MappingProxyType)
        and isinstance(sharegpt_record.payload["messages"], tuple)
        and all(isinstance(message, MappingProxyType) for message in sharegpt_record.payload["messages"])
    ),
    "preference_pair_is_atomic": (
        set(preference_payload) == {"prompt", "chosen", "rejected"}
        and preference_payload["chosen"] != preference_payload["rejected"]
    ),
    "reasoning_fields_are_separate": (
        {"problem", "reasoning_trace", "final_answer", "step_labels", "verifier_revision"}
        == set(reasoning_payload)
    ),
    "reasoning_verifier_matches_contract": (
        reasoning_payload["verifier_revision"]
        == reasoning_record.metadata["reward_or_verifier_revision"]
    ),
    "record_and_split_group_identity_are_separate": (
        reasoning_record.split_group_id == "problem-001"
    ),
    "corpus_uses_canonical_path": (
        len(corpus_canonical_records) + len(corpus_quarantine_records) == len(documents)
        and {record.record_id for record in corpus_canonical_records}
        == {document.document_id for document in canonical_documents}
    ),
    "visible_source_text_hashes_match": all(
        record.lineage["source_text_sha256"]
        == hashlib.sha256(str(record.payload["text"]).encode("utf-8")).hexdigest()
        for record in corpus_canonical_records
    ),
    "split_groups_do_not_cross_partitions": all(
        len(memberships) == 1 for memberships in split_membership_by_group.values()
    ),
    "contract_policies_match_pipeline": (
        dataset_contracts["web-corpus-v1"].quality_policy == QUALITY_POLICY_REVISION
        and dataset_contracts["web-corpus-v1"].license_policy == LICENSE_POLICY_REVISION
    ),
    "invalid_record_is_quarantined": (
        len(quarantine_records) == 1
        and quarantine_records[0]["detail"] == "required_non_empty_text:instruction"
        and len(quarantine_records[0]["input_record_sha256"]) == 64
        and quarantine_records[0]["schema_registry_revision"] == SCHEMA_REGISTRY_REVISION
    ),
    "lineage_is_complete": all(
        len(record.lineage["input_record_sha256"]) == 64
        and len(record.lineage["canonical_payload_sha256"]) == 64
        and bool(record.lineage["source_uri"])
        and record.lineage["schema_registry_revision"] == SCHEMA_REGISTRY_REVISION
        and bool(record.lineage["adapter_revision"])
        for record in canonical_records
    ),
}
print(schema_contract_checks)


## 5．迁移到生产库

| 阶段 | 并行方式 | 主要 Shuffle | 推荐恢复边界 |
|---|---|---|---|
| 拉取与解析 | 按源文件/对象分区 | 无 | 原始快照与解析结果 |
| Decode/Schema 校验 | 按对象或 Row Group 分区 | 无 | Reader + Schema Registry Revision + Quarantine |
| Canonicalize/目标视图物化 | Record Map；按 `split_group_key` 保持原子性 | 通常无 | Canonical Partition + Adapter/Materializer Revision |
| 语言/质量/PII | Document Map | 无 | 策略版本 + 分区结果 |
| 精确去重 | Hash Partition | 内容哈希 | Hash Bucket |
| MinHash/LSH | Signature + Bucket | LSH Bucket | 候选桶 |
| Split/混合 | Group Map + 统计聚合 | 来源/Schema/内容标签统计 | Split Manifest |
| Template/Tokenize | Group Map | 通常无 | Template + Tokenizer/Processor + Label Policy |
| Block/Packing | 有序流与边界合并 | 受 Packing 策略影响 | 完整 Block + Canonical Source Span |
| 构建规格寻址发布 | Build 级暂存、校验与原子目录提交 | 无 | 不可变 Build Spec ID + Release Manifest 摘要 + Shard Hash |

JSON Schema 或 Pydantic 适合校验 JSON-facing Contract，PyArrow Schema 与 Parquet/Arrow IPC 适合类型化列式存储，Hugging Face `Features` 适合训练侧数据接口；这些库对象都需要由同一份 Schema Registry、Adapter Revision 和 Golden Record 测试生成或校验，不能各自维护一套字段定义。

Spark、Ray Data、DataTrove 或其他引擎负责调度和容错；数据语义仍由本章的 Manifest、Schema Registry、策略版本和阶段契约定义。执行框架的临时目录不作为唯一资产来源。JSONL 适合作为交换格式和可追溯落地区；大规模扫描、嵌套类型与统计分析通常物化为带显式 Schema 的 Parquet/Arrow IPC，权威性由 Manifest 和血缘决定，而不是由扩展名决定。


## 6．生产边界

1. Dataset Contract 固定容器格式、压缩与编码、Schema 名称与版本、样本类型、消费用途、`record_id_key`、`split_group_key`、模态、Adapter、Label Policy 和许可证策略。
2. 未知、歧义或校验失败的 Schema 不静默猜测；记录稳定原因码、来源定位、已解析输入哈希和 Registry Revision 后进入受控隔离区。普通 Manifest 只发布聚合计数和受控记录级审计资产的摘要/引用，不直接泄露隔离原文或敏感来源。
3. 每个来源都有实际许可证、授权用途、保留期限、负责人、快照时间和准确命名的源对象/源文本哈希；混合来源同时绑定许可证准入策略。
4. 每个阶段按来源、Schema、内容标签和模态记录输入/输出样本数、字节数、Token 数、原因码和放大率。
5. 精确/近似去重、质量和污染阈值使用按样本类型分层的人工标注集校准。
6. Split 在 Contract 的 `split_group_key` 级稳定；会话、偏好对、工具轨迹、问题家族及其媒体不会跨集合。
7. 原始 CoT、可验证证据和最终答案分别保存并设置访问、脱敏与保留策略；原始 CoT 不作为普通 API 输出或审计日志，训练细节由 `A30` 承担。
8. 目标温度权重、符合准入的单遍文档权重和实际持久化 Token 权重分别记录；未应用的目标权重不得标成 Shard 实际分布。
9. Chat Template、Tokenizer/Processor ID 与文件哈希、特殊 Token、EOS、Label Mask、Packing、尾部处理、有效策略参数、代码 Revision 与关键运行库版本和数据版本共同冻结；不得把 T5 Sentinel 临时改作 BOS。
10. JSONL 可以作为权威交换资产；面向大规模内部处理物化 Parquet/Arrow IPC 时保留逐条来源哈希与可逆映射，不能用格式转换切断血缘。
11. 分区任务可重入；本地文件系统在同一挂载点内完成 Build 级暂存、校验与原子目录提交，已发布的构建规格寻址目录不覆盖；并发发布者复用并复核胜者。对象存储使用不可变对象键、完成标记或事务目录，并处理并发发布；逐 Shard SHA-256 与 Release Manifest 摘要负责完整性，外部制品目录固定摘要或 KMS 签名负责真实性。
12. 监控吞吐、尾延迟、Schema 漂移、隔离率、来源/Token 分布漂移、对象存储错误、Shuffle Spill、热点 Bucket 和成本。
13. 删除请求能够通过 Shard 中结构化的 `dataset_id` + `record_id` 键和 Block `source_spans` 沿血缘定位所有 Canonical Partition、目标视图、派生 Shard 和训练产物，并触发治理流程。

规模化数据工程的验收目标不止于文件生成成功，而是训练系统能消费一份可解释、可复现、可撤回且质量经过验收的数据资产。
